In [1]:
%load_ext watermark
%watermark -a "Manuel Elias Orellana Lavayen" -d -v -iv

Author: Manuel Elias Orellana Lavayen

Date: 2026-08-26

Python implementation: CPython
Python version       : 3.12.10
IPython version      : 9.16.1



# Librerias

In [2]:
import pandas as pd
import re
import spacy
from symspellpy import SymSpell
import numpy as np
from pathlib import Path
ruta_base = Path.cwd() #Ruta base

##### Uso de GPU

In [3]:
import torch
print(torch.cuda.is_available())
print()

import cupy
print(cupy.cuda.runtime.getDeviceCount())
print(cupy.cuda.Device())
print()

print("GPU disponible:", spacy.prefer_gpu())
print("GPU en uso:", spacy.require_gpu())

True

1
<CUDA Device 0>

GPU disponible: True
GPU en uso: True


# Datos

In [4]:
df_train= pd.read_csv(ruta_base/ "Datasets/train.csv", usecols= ["text", "label"])
df_validation = pd.read_csv(ruta_base / "Datasets/validation.csv", usecols= ["text", "label"])
df_test = pd.read_csv(ruta_base / "Datasets/test.csv", usecols= ["text", "label"])

## Juntando Clases

In [5]:
juntar = {
    0: 0,
    1: 0,
    2: 1,
    3: 2,
    4: 2
}
df_train["label"] = df_train["label"].map(juntar)
df_validation["label"] = df_validation["label"].map(juntar)
df_test["label"] = df_test["label"].map(juntar)

## Informacion Datasets

In [6]:
datasets = {"train": df_train, "validation": df_validation, "test": df_test}

In [7]:
# Distribución de etiquetas de datasets
print("=========Distribución de datos=======\n")
for nombre_dataset, dataset in datasets.items():
    print(f"---{nombre_dataset}:")
    print(f"Número de etiquetas: {dataset['label'].nunique()}")
    print(f"{list(dataset['label'].value_counts(normalize=True) * 100)}")

# Dimensiones de datasets
print("\n=========Dimensiones de datasets=======\n")
for nombre_dataset, dataset in datasets.items():
    print(f"---{nombre_dataset:<12}: {dataset.shape}")

=========Distribución de datos=======

---train:
Número de etiquetas: 3
[40.0, 40.0, 20.0]
---validation:
Número de etiquetas: 3
[40.0, 40.0, 20.0]
---test:
Número de etiquetas: 3
[40.0, 40.0, 20.0]

=========Dimensiones de datasets=======

---train       : (200000, 2)
---validation  : (5000, 2)
---test        : (5000, 2)


# Limpieza Dataset

#### Exploración

In [8]:
# Valores nulos del dataset
print("=========Valores nulos datasets=======\n")
for nombre_dataset, dataset in datasets.items():
    print(f"---{nombre_dataset}:")
    print(f"{(dataset.isnull().sum()).to_dict()}")

# Duplicados del dataset
print("\n=========Valores Duplicados datasets=======\n")
for nombre_dataset, dataset in datasets.items():
    print(f"---{nombre_dataset:<12}: {dataset.duplicated().sum()}")

=========Valores nulos datasets=======

---train:
{'text': 0, 'label': 0}
---validation:
{'text': 0, 'label': 0}
---test:
{'text': 0, 'label': 0}

=========Valores Duplicados datasets=======

---train       : 1524
---validation  : 4
---test        : 6


#### Acciones

In [9]:
# Eliminación de duplicados
for nombre_dataset, dataset in datasets.items():
    print(f"---Eliminando duplicados de {nombre_dataset}")
    datasets[nombre_dataset] = datasets[nombre_dataset].drop_duplicates()

---Eliminando duplicados de train
---Eliminando duplicados de validation
---Eliminando duplicados de test


## Crear Columna con textos corregidos ortograficamente

### Configuración de Corrector Ortográfico

In [10]:
sym_spell = SymSpell(max_dictionary_edit_distance=1, prefix_length=7)
ruta_diccionario_ortografia = ruta_base / "Recursos de procesamiento de texto/es-100l.txt"
sym_spell.load_dictionary(ruta_diccionario_ortografia, term_index=0, count_index=1, separator=" ")

True

#### Función para corregir ortografia de textos

In [11]:
def corregir_ortografia(texto):
    """
    Corregir ortografia de un texto

    Args:
        texto: texto a limpiar

    Returns:
        texto corregido ortograficamente
    """

    if pd.isna(texto) or not str(texto).strip():
        return texto

    palabras = texto.split()
    palabras_corregidas = []

    for palabra in palabras:

        # Separar puntuación del inicio y final
        match = re.match(r"^([^a-záéíóúüñ]*)([a-záéíóúüñ]+)([^a-záéíóúüñ]*)$",palabra.lower())

        # Si no es una palabra normal, mantenerla
        if not match:
            palabras_corregidas.append(palabra)
            continue

        inicio, palabra_limpia, final = match.groups()

        # Si la palabra ya está en el diccionario, NO corregirla
        if palabra_limpia in sym_spell.words:
            palabras_corregidas.append(inicio + palabra_limpia + final)
            continue

        # Si no existe, buscar una posible corrección
        sugerencias = sym_spell.lookup(palabra_limpia,verbosity=0,max_edit_distance=1)

        if sugerencias:
            palabra_corregida = sugerencias[0].term
        else:
            palabra_corregida = palabra_limpia

        palabras_corregidas.append(
            inicio + palabra_corregida + final
        )

    return " ".join(palabras_corregidas)

In [12]:
for nombre_dataset, dataset in datasets.items():
    print(f"---Correción Ortografica: {nombre_dataset}")
    datasets[nombre_dataset]["text_ortografia"] = (datasets[nombre_dataset]["text"].apply(corregir_ortografia))

---Correción Ortografica: train
---Correción Ortografica: validation
---Correción Ortografica: test


# Limpieza y normalización Texto

### Recurso local de Spacy para procesamiento de texto

In [13]:
#Cargando Recurso de Spacy para Limpieza de textos
ruta_modelo_spacy = ruta_base / "Recursos de procesamiento de texto/es_core_news_md"
spacy.prefer_gpu() #indicar a Spacy que use la GPU

nlp = spacy.load(
    ruta_modelo_spacy,
    disable=["parser", "ner"] # Descartar Análisis sintáctico de dependencias Y Reconocimiento de entidades
)

##### Stop Words de Spacy (Conservando negaciones o palabras que muestren sentimientos)

In [14]:
stop_words = nlp.Defaults.stop_words.copy()

remover = {
    # negaciones
    "no", "nunca", "jamás", "sin", "ni", "nadie",

    # palabras de sentimiento
    "bueno", "buena", "buenos",
    "malo", "mal",
    "mejor", "peor"
}

stop_words.difference_update(remover)

### Funciones limpieza

In [15]:
#Limpieza Básica
def limpiar_texto(texto):
    """
    Limpiar un texot de etiquetas HTML, URL, espacios extra y lo convierte a minusculas

    Args:
        texto: texto

    Returns:
        texto limpio
    """

    # Minúsculas
    texto = texto.lower()

    # Eliminar HTML
    texto = re.sub(r"<.*?>", " ", texto)

    # Eliminar URL
    texto = re.sub(r"http\S+|www\S+", "", texto)

    # Eliminar espacios extra
    texto = " ".join(texto.split())

    return texto

# Eliminación de Stop words y lematización
def limpiar_texto_tfidf (dataframe: pd.DataFrame, ortografia = False):
    """
    Limpia el texto, además aplica lematizacion y eliminacion de stop words

    Args:
        dataframe: Dataframe que tenga los textos
        ortografia: Booleano que indique si se va limpiar texto con correccion ortografica

    Returns:
        lista con texto limpio
    """

    if ortografia:
        textos_limpios = dataframe["text_ortografia"].apply(limpiar_texto)
    else:
        textos_limpios = dataframe["text"].apply(limpiar_texto)

    docs = nlp.pipe(textos_limpios, batch_size=4000)

    resultado = []

    for doc in docs:

        tokens = []

        for token in doc:

            if token.lemma_ not in stop_words:
                tokens.append(token.lemma_)

        texto_resultado = " ".join(tokens)

        if not texto_resultado:
            texto_resultado = np.nan

        resultado.append(texto_resultado)

    return resultado

### Aplicando funciones de limpieza

In [16]:
for nombre_dataset, dataset in datasets.items():
    print(f"---{nombre_dataset}")
    print("Procesando para TF-IDF")
    #Correcion Ortografica Inactiva
    print("Normal")
    texto_limpio_tfidf = limpiar_texto_tfidf(dataset)
    dataset["text_tfidf"] = texto_limpio_tfidf
    #Correccion ortografica activa
    print("Correcion Ortografica")
    texto_limpio_tfidf_ortografia = limpiar_texto_tfidf(dataset, ortografia = True)
    dataset["text_tfidf_ortografia"] = texto_limpio_tfidf_ortografia
    print("Procesando para Embeddings")
    print("Normal")
    texto_limpio_embeddings = dataset["text"].apply(limpiar_texto)
    dataset["text_embedding"] = texto_limpio_embeddings
    #Correcion Ortografica activa
    print("Correcion Ortografica")
    texto_limpio_embeddings_ortografia = dataset["text_ortografia"].apply(limpiar_texto)
    dataset["text_embedding_ortografia"] = texto_limpio_embeddings_ortografia

---train
Procesando para TF-IDF
Normal
Correcion Ortografica
Procesando para Embeddings
Normal
Correcion Ortografica
---validation
Procesando para TF-IDF
Normal
Correcion Ortografica
Procesando para Embeddings
Normal
Correcion Ortografica
---test
Procesando para TF-IDF
Normal
Correcion Ortografica
Procesando para Embeddings
Normal
Correcion Ortografica


# Tablas Resultantes

In [17]:
pd.set_option('display.max_colwidth', None)
datasets["train"].head()

,text,label,text_ortografia,text_tfidf,text_tfidf_ortografia,text_embedding,text_embedding_ortografia
0,Nada bueno se me fue ka pantalla en menos de 8 meses y no he recibido respuesta del fabricante,0,nada bueno se me fue ka pantalla en menos de 8 meses y no he recibido respuesta del fabricante,bueno ka pantalla 8 mes no recibir respuesta fabricante,bueno ka pantalla 8 mes no recibir respuesta fabricante,nada bueno se me fue ka pantalla en menos de 8 meses y no he recibido respuesta del fabricante,nada bueno se me fue ka pantalla en menos de 8 meses y no he recibido respuesta del fabricante
1,"Horrible, nos tuvimos que comprar otro porque ni nosotros que sabemos inglés, ni un informático, después de una hora fue capaz de instalarlo",0,"horrible, nos tuvimos que comprar otro porque ni nosotros que sabemos inglés, ni un informático, después de una hora fue capaz de instalarlo","horrible , comprar ni inglés , ni informático , hora capaz instalar él","horrible , comprar ni inglés , ni informático , hora capaz instalar él","horrible, nos tuvimos que comprar otro porque ni nosotros que sabemos inglés, ni un informático, después de una hora fue capaz de instalarlo","horrible, nos tuvimos que comprar otro porque ni nosotros que sabemos inglés, ni un informático, después de una hora fue capaz de instalarlo"
2,"Te obligan a comprar dos unidades y te llega solo una y no hay forma de reclamar, una autentica estafa, no compreis!!",0,"te obligan a comprar dos unidades y te llega solo una y no hay forma de reclamar, una autentica estafa, no compres!!","obligar comprar unidad llegar no forma reclamar , autentico estafa , no compreis ! !","obligar comprar unidad llegar no forma reclamar , autentico estafa , no comprar ! !","te obligan a comprar dos unidades y te llega solo una y no hay forma de reclamar, una autentica estafa, no compreis!!","te obligan a comprar dos unidades y te llega solo una y no hay forma de reclamar, una autentica estafa, no compres!!"
3,"No entro en descalificar al vendedor, solo puedo decir que tras dos meses de espera.... sigo sin el producto y tuve que contactar con Amazon para reclamar su reembolso. Amazon un 10 . Se hace cargo del problema, pero yo e desembolsado mi dinero y en dos meses me lo devuelven Perdida de tiempo TOTAL. Sin palabras. Y Ustedes deciden",0,"no entro en descalificar al vendedor, solo puedo decir que tras dos meses de espera.... sigo sin el producto y tuve que contactar con amaron para reclamar su reembolso. amaron un 10 . se hace cargo del problema, pero yo e desembolsado mi dinero y en dos meses me lo devuelven perdida de tiempo total. sin palabras. y ustedes deciden","no entro descalificar vendedor , mes espera .... seguir sin producto contactar amazon reclamar reembolso . amazon 10 . cargo problema , desembolsar dinero mes devolver perdido tiempo . sin palabra . decidir","no entro descalificar vendedor , mes espera .... seguir sin producto contactar amar reclamar reembolso . amar 10 . cargo problema , desembolsar dinero mes devolver perdido tiempo . sin palabra . decidir","no entro en descalificar al vendedor, solo puedo decir que tras dos meses de espera.... sigo sin el producto y tuve que contactar con amazon para reclamar su reembolso. amazon un 10 . se hace cargo del problema, pero yo e desembolsado mi dinero y en dos meses me lo devuelven perdida de tiempo total. sin palabras. y ustedes deciden","no entro en descalificar al vendedor, solo puedo decir que tras dos meses de espera.... sigo sin el producto y tuve que contactar con amaron para reclamar su reembolso. amaron un 10 . se hace cargo del problema, pero yo e desembolsado mi dinero y en dos meses me lo devuelven perdida de tiempo total. sin palabras. y ustedes deciden"
4,Llega tarde y co la talla equivocada,0,llega tarde y con la talla equivocada,llegar co talla equivocado,llegar talla equivocado,llega tarde y co la talla equivocada,llega tarde y con la talla equivocada


In [18]:
for nombre_dataset, dataset in datasets.items():
    datasets[nombre_dataset] = (datasets[nombre_dataset].dropna().drop_duplicates().reset_index(drop=True))

In [19]:
for nombre_dataset, dataset in datasets.items():
    print(f"\n===== {nombre_dataset} =====")

    for columna in dataset.columns:
        vacios = (
            dataset[columna]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        )

        print(f"{columna:<30}: {vacios} cadenas vacías")


===== train =====
text                          : 0 cadenas vacías
label                         : 0 cadenas vacías
text_ortografia               : 0 cadenas vacías
text_tfidf                    : 0 cadenas vacías
text_tfidf_ortografia         : 0 cadenas vacías
text_embedding                : 0 cadenas vacías
text_embedding_ortografia     : 0 cadenas vacías

===== validation =====
text                          : 0 cadenas vacías
label                         : 0 cadenas vacías
text_ortografia               : 0 cadenas vacías
text_tfidf                    : 0 cadenas vacías
text_tfidf_ortografia         : 0 cadenas vacías
text_embedding                : 0 cadenas vacías
text_embedding_ortografia     : 0 cadenas vacías

===== test =====
text                          : 0 cadenas vacías
label                         : 0 cadenas vacías
text_ortografia               : 0 cadenas vacías
text_tfidf                    : 0 cadenas vacías
text_tfidf_ortografia         : 0 cadenas vacías
text_emb

In [20]:
for nombre_dataset, dataset in datasets.items():
    print(f"\n===== {nombre_dataset} =====")

    for columna in dataset.columns:
        vacios = (
            dataset[columna]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        )

        if vacios > 0:
            print(f"{columna}: {vacios}")


===== train =====

===== validation =====

===== test =====


# Guardar Datasets Limpios en CSV

In [21]:
dataset_train = datasets["train"].copy()
dataset_validation = datasets["validation"].copy()
dataset_test = datasets["test"].copy()

In [22]:
print("Train:", dataset_train.isna().sum().sum())
print("Validation:", dataset_validation.isna().sum().sum())
print("Test:", dataset_test.isna().sum().sum())

Train: 0
Validation: 0
Test: 0


In [23]:
dataset_train.to_csv("./Datasets Limpios y corregidos/dataset_train.csv", index=False)
dataset_validation.to_csv("./Datasets Limpios y corregidos/dataset_validation.csv", index=False)
dataset_test.to_csv("./Datasets Limpios y corregidos/dataset_test.csv", index=False)